# Inventory Domain Walkthrough

This notebook uses the service layer directly instead of shelling out to the CLI. That keeps the examples short while still exercising the same domain workflows the CLI uses.


In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root. Start Jupyter from somewhere inside the repo.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
FIXTURES_DIR = REPO_ROOT / "tests" / "fixtures"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


def reset_dir(path: Path) -> Path:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def copy_fixture_bundle(destination: Path, fixture_name: str, *filenames: str) -> dict[str, Path]:
    destination.mkdir(parents=True, exist_ok=True)
    copied: dict[str, Path] = {}
    for filename in filenames:
        source = FIXTURES_DIR / fixture_name / filename
        target = destination / filename
        shutil.copyfile(source, target)
        copied[filename] = target
    return copied

from decimal import Decimal

from mtg_source_stack.db.connection import connect
from mtg_source_stack.db.schema import initialize_database
from mtg_source_stack.importer.mtgjson import import_mtgjson_identifiers, import_mtgjson_prices
from mtg_source_stack.importer.scryfall import import_scryfall_cards
from mtg_source_stack.inventory.csv_import import import_csv
from mtg_source_stack.inventory.response_models import serialize_response
from mtg_source_stack.inventory.service import (
    add_card,
    create_inventory,
    list_owned_filtered,
    merge_rows,
    search_cards,
    set_location,
    set_quantity,
    set_tags,
    split_row,
)

WORKDIR = reset_dir(REPO_ROOT / "var" / "walkthrough" / "03_inventory_domain")
DB_PATH = WORKDIR / "collection.db"
fixture_bundle = copy_fixture_bundle(
    WORKDIR,
    "lightning_bolt",
    "scryfall.json",
    "identifiers.json",
    "prices.json",
    "inventory_import.csv",
)
initialize_database(DB_PATH)
import_scryfall_cards(DB_PATH, fixture_bundle["scryfall.json"])
import_mtgjson_identifiers(DB_PATH, fixture_bundle["identifiers.json"])
import_mtgjson_prices(DB_PATH, fixture_bundle["prices.json"])
create_inventory(DB_PATH, slug="personal", display_name="Personal Collection", description="Notebook demo")

def pretty(value):
    print(json.dumps(serialize_response(value), indent=2))

def owned_rows():
    return list_owned_filtered(
        DB_PATH,
        inventory_slug="personal",
        provider="tcgplayer",
        limit=None,
        query=None,
        set_code=None,
        rarity=None,
        finish=None,
        condition_code=None,
        language_code=None,
        location=None,
        tags=None,
    )

print(f"Working directory: {WORKDIR}")


## Search The Local Catalog

Search is local-only here: it queries the imported SQLite catalog instead of reaching out to Scryfall live.


In [ ]:
search_results = search_cards(DB_PATH, query="Lightning Bolt", exact=False, limit=5)
pretty(search_results)


## Add A Card And Inspect Owned Rows

We will use the first matching printing from the local catalog and create an owned inventory row.


In [ ]:
selected_card = search_results[0]
added = add_card(
    DB_PATH,
    inventory_slug="personal",
    inventory_display_name=None,
    scryfall_id=selected_card.scryfall_id,
    tcgplayer_product_id=None,
    name=None,
    set_code=None,
    collector_number=None,
    lang=None,
    quantity=2,
    condition_code="NM",
    finish="normal",
    language_code="en",
    location="Binder A",
    acquisition_price=Decimal("1.25"),
    acquisition_currency="USD",
    notes="Notebook demo row",
    tags="burn,trade",
)
pretty(added)
print("\nOwned rows after add:")
pretty(owned_rows())


## Mutate The Inventory Row

The service layer supports field-level edits plus split and merge operations.


In [ ]:
set_quantity(DB_PATH, inventory_slug="personal", item_id=added.item_id, quantity=3)
set_location(DB_PATH, inventory_slug="personal", item_id=added.item_id, location="Deck Box")
set_tags(DB_PATH, inventory_slug="personal", item_id=added.item_id, tags="burn,featured")
split_result = split_row(
    DB_PATH,
    inventory_slug="personal",
    item_id=added.item_id,
    quantity=1,
    condition_code=None,
    finish=None,
    language_code=None,
    location="Trade Binder",
)
merge_result = merge_rows(
    DB_PATH,
    inventory_slug="personal",
    source_item_id=split_result.item_id,
    target_item_id=added.item_id,
)
print("Split result:")
pretty(split_result)
print("\nMerge result:")
pretty(merge_result)
print("\nOwned rows after mutations:")
pretty(owned_rows())


## Preview A CSV Import

Dry-run mode reuses the real import path and rolls the transaction back at the end so the preview stays faithful without changing the database.


In [ ]:
preview = import_csv(
    DB_PATH,
    csv_path=fixture_bundle["inventory_import.csv"],
    default_inventory="personal",
    dry_run=True,
)
pretty(preview)


## Inspect The Audit Trail

Every mutation writes rows into `inventory_audit_log`, including split and merge operations.


In [ ]:
with connect(DB_PATH) as connection:
    audit_rows = [dict(row) for row in connection.execute(
        """
        SELECT id, action, item_id, actor_type, occurred_at, before_json, after_json
        FROM inventory_audit_log
        ORDER BY id DESC
        LIMIT 10
        """
    ).fetchall()]

print(json.dumps(audit_rows, indent=2))


## Next Step

Open `04_reporting_and_api_contract_walkthrough.ipynb` to see how the same service layer feeds reports, exports, and API-facing payloads.
